In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
MAX_WORDS = 20000
MAX_LEN   = 200
EMB_DIM   = 128

In [5]:
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=MAX_WORDS)
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=MAX_LEN)
x_test  = keras.preprocessing.sequence.pad_sequences(x_test,  maxlen=MAX_LEN)


In [6]:
from keras.models import Sequential
from keras.layers import Input, LSTM, Dense
from keras import Input
from keras import layers, models

In [9]:
inputs = Input(shape=(MAX_LEN,), dtype="int32", name="tokens")
x = layers.Embedding(MAX_WORDS, EMB_DIM, mask_zero=True)(inputs)
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64))(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ tokens (InputLayer) │ (None, 200)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 128)  │  2,560,000 │ tokens[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 200)       │          0 │ tokens[0][0]      │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 200, 128)  │     98,816 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 128)       │     98,816 │ bidirectional[0]… │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │        129 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,757,761 (10.52 MB)

 Trainable params: 2,757,761 (10.52 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])

In [ ]:
history = model.fit(x_train, y_train, batch_size=256, epochs=3,
                    validation_split=0.2, verbose=2)

Epoch 1/3
79/79 - 343s - 4s/step - accuracy: 0.9933 - loss: 0.0203 - val_accuracy: 0.8676 - val_loss: 0.6534
Epoch 2/3
79/79 - 346s - 4s/step - accuracy: 0.9985 - loss: 0.0067 - val_accuracy: 0.8640 - val_loss: 0.7434
Epoch 3/3
79/79 - 344s - 4s/step - accuracy: 0.9991 - loss: 0.0029 - val_accuracy: 0.8662 - val_loss: 0.6878


In [23]:

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"IMDB Test Accuracy: {test_acc:.4f}")

IMDB Test Accuracy: 0.8544
